# Lab 02 — The Agent Loop & Design Patterns

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives.** After this lab you can:

- implement the **minimal agent loop** (reason → act → observe → decide) from scratch around `ollama.chat`, with the message list as the agent's *entire* working state,
- implement and justify explicit **stop conditions**: final-answer detection, a step cap, and a budget cap,
- dispatch model-emitted **tool calls** to plain Python functions and feed results — including errors — back as observations,
- **instrument** the loop: log every iteration and measure how resending the full history produces the *quadratic token tax*,
- reproduce the three canonical **failure modes** — infinite loops, context overflow, derailment — and apply first-line defences (duplicate-call detection, truncation, goal re-injection),
- place the loop in Ng's taxonomy of the **four agentic design patterns** (Planning, Tool Use, Reflection, Multi-Agent).

## Theory recap: one loop, one growing message list

### The loop is the agent

Session 02's central claim: **an agent is a while-loop wrapped around a stateless LLM.** One iteration has four phases (the ReAct formulation, Yao et al., 2023):

1. **Reason** — the runtime sends the *entire* message list to the model; the model returns either plain text or a structured tool call. Nothing else is "in its head".
2. **Act** — the *runtime* (your plain Python process) parses the tool call and executes the corresponding function, with the runtime's real privileges. The model never executes anything.
3. **Observe** — the tool result is appended to the message list as a new message; the model only sees it on the *next* call.
4. **Decide** — implicit: there is no explicit branch in the model. A tool call means "loop again"; plain text means "done". The output type itself is the control signal.

**The model proposes; the runtime disposes.** Capabilities and risks live in the runtime, not the model.

### State = the growing message list

The LLM is a pure function: context in, tokens out, no memory between API calls. The agent's entire working state — the task, every action tried, every observation, every intermediate conclusion — lives in the **message list** that the loop maintains. Consequence: the agent is trivially *serialisable* — dump the list to JSON, restore it tomorrow on another machine, resume the loop. Crash recovery, audit trails and replay debugging all fall out of this.

### Stop conditions

Four families, each a different judgement about *when to end*:

| Stop condition | Whose judgement ends the run? |
|---|---|
| Final answer | the model's (task looks complete) |
| Step cap | the engineer's (prior bound on iterations) |
| Budget cap | operational/financial (tokens, money, time) |
| Human interrupt | the operator's (approval gates, kill switch) |

The minimal agent implements only the first two; this lab adds the third and first-line defences.

### Three canonical failure modes

- **Infinite loop** — the same call with the same arguments, repeated in hope of a different result. No error is raised anywhere.
- **Context overflow** — the list outgrows the context window: hard API errors, or *silent* eviction/truncation of the oldest tokens — often including the original task.
- **Derailment** — each step locally reasonable, the trajectory globally off-goal; the goal is one line, thousands of tokens back, while recent observations dominate attention.

None of these is a bug in the model. They are **emergent properties of iteration** — the unit of failure is the *trajectory*, not the call.

### Token economics: the quadratic tax

If each iteration adds ≈ $k$ tokens and the full history is resent every call, the input at step $i$ is ≈ $i \cdot k$, so cumulative input over $n$ steps is $k \, n(n+1)/2 = \Theta(n^2)$. With $k = 2000$ and $n = 20$: ≈ 420,000 cumulative input tokens for a final context of only 40,000. Levers: prompt caching, observation hygiene (truncate/filter), compaction, model routing, budget caps.

### Four agentic design patterns (Ng, 2024)

**Planning** (decompose into subgoals, execute against the plan — counters derailment), **Tool Use** (external actions — the pattern our minimal agent already implements), **Reflection** (draft → review → redraft), **Multi-Agent** (specialised roles with separate contexts). Each is a disciplined *token-for-quality trade*; none of them changes the loop itself.

### This lab

The lecture's lab announcement: *"You implement exactly this loop, then break it: remove the step cap, feed it oversized pages, and watch each failure mode appear."* Our running example — the **research agent** — works against a small *offline* mini-web in `data/`, so no live internet access is needed.

## Part A — Setup

Dependencies: `ollama`, `numpy`, `pandas`, `matplotlib` (optionally `ipywidgets` for Part G).
Install the Python client with `pip install ollama` if needed. Any tool-capable 7–30B model
works; we default to `qwen2.5:7b` (use `qwen2.5:3b` if you are short on RAM — set the
`OLLAMA_MODEL` environment variable). A recent `ollama` Python package (≥ 0.4) is assumed.

In [ ]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ollama

MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")
print("Using model:", MODEL)

In [ ]:
# Connectivity check — run this before anything else.
try:
    ollama.chat(model=MODEL,
                messages=[{"role": "user", "content": "Reply with the single word: ready"}])
    print(f"Ollama is reachable and '{MODEL}' responds - you are good to go.")
except Exception as e:
    print("Could not reach Ollama:", e)
    print("Start Ollama with `ollama serve` and pull the model with `ollama pull qwen2.5:7b`.")

## Part B — The tool: a tiny offline web

The research agent needs a web to search. We simulate one: `data/search_corpus.json`
contains seven small synthetic pages (EU AI Act material, one derailment bait page on US
policy, one noise page, and one deliberately *gigantic* page for Part E). Two plain Python
functions act as our tools:

- `web_search(query)` — keyword search, returns the top-3 pages as **300-character snippets**,
- `read_page(page_id)` — returns the **full** text of one page.

Remember the lecture: *the model never sees the Python function* — only the JSON schema you
declare in the next cell. Interface design is prompt design.

Fill the gaps (three underscores) in the next two cells.

In [ ]:
# The research agent's "web": a small offline corpus of synthetic pages.
with open("data/search_corpus.json", encoding="utf-8") as f:
    CORPUS = json.load(f)   # {page_id: {"title": ..., "content": ...}}

print(f"{len(CORPUS)} pages in the mini-web:")
for pid, page in CORPUS.items():
    print(f"  [{pid:>23}]  {page['title'][:60]}  ({len(page['content'])} chars)")

STOPWORDS = {"the", "and", "for", "what", "are", "does", "with", "that",
             "this", "from", "how", "into", "was", "were", "has", "have"}

def keywords(text):
    """Lower-cased words, punctuation stripped, stopwords and short words removed."""
    words = [w.strip(".,?!:;\"'()[]").lower() for w in text.split()]
    return [w for w in words if len(w) > 2 and w not in STOPWORDS]

def web_search(query: str) -> str:
    """Search the mini-web; return the top-3 hits as short snippets."""
    q_words = keywords(query)
    scores = {}
    for pid, page in CORPUS.items():
        page_words = keywords(page["title"] + " " + page["content"])
        scores[pid] = sum(page_words.count(w) for w in ___)   # count keyword hits
    top = sorted(scores, key=scores.get, reverse=___)[:3]     # best three pages first
    top = [pid for pid in top if scores[pid] > 0]
    if not top:
        return "No results found."
    return "\n\n".join(
        f"[{pid}] {CORPUS[pid]['title']}\n{CORPUS[pid]['content'][:300]} ..."
        for pid in top)

def read_page(page_id: str) -> str:
    """Return the FULL text of one page (can be very large!)."""
    if page_id not in CORPUS:
        return f"TOOL ERROR: unknown page_id '{page_id}' - use web_search to find valid page_ids."
    return CORPUS[___]["content"]

print()
print(web_search("EU AI Act obligations general-purpose models")[:600])

<details>
<summary><b>Click here for the solution</b></summary>

```python
# The research agent's "web": a small offline corpus of synthetic pages.
with open("data/search_corpus.json", encoding="utf-8") as f:
    CORPUS = json.load(f)   # {page_id: {"title": ..., "content": ...}}

print(f"{len(CORPUS)} pages in the mini-web:")
for pid, page in CORPUS.items():
    print(f"  [{pid:>23}]  {page['title'][:60]}  ({len(page['content'])} chars)")

STOPWORDS = {"the", "and", "for", "what", "are", "does", "with", "that",
             "this", "from", "how", "into", "was", "were", "has", "have"}

def keywords(text):
    """Lower-cased words, punctuation stripped, stopwords and short words removed."""
    words = [w.strip(".,?!:;\"'()[]").lower() for w in text.split()]
    return [w for w in words if len(w) > 2 and w not in STOPWORDS]

def web_search(query: str) -> str:
    """Search the mini-web; return the top-3 hits as short snippets."""
    q_words = keywords(query)
    scores = {}
    for pid, page in CORPUS.items():
        page_words = keywords(page["title"] + " " + page["content"])
        scores[pid] = sum(page_words.count(w) for w in q_words)   # count keyword hits
    top = sorted(scores, key=scores.get, reverse=True)[:3]     # best three pages first
    top = [pid for pid in top if scores[pid] > 0]
    if not top:
        return "No results found."
    return "\n\n".join(
        f"[{pid}] {CORPUS[pid]['title']}\n{CORPUS[pid]['content'][:300]} ..."
        for pid in top)

def read_page(page_id: str) -> str:
    """Return the FULL text of one page (can be very large!)."""
    if page_id not in CORPUS:
        return f"TOOL ERROR: unknown page_id '{page_id}' - use web_search to find valid page_ids."
    return CORPUS[page_id]["content"]

print()
print(web_search("EU AI Act obligations general-purpose models")[:600])
```

</details>

In [ ]:
# What the model actually sees: JSON schemas, never the Python bodies.
TOOLS = [
    {"type": "function",
     "function": {
         "name": "web_search",
         "description": ("Search an offline copy of the web. Returns the top-3 matching "
                         "pages as [page_id], title, and a 300-character snippet. "
                         "Snippets are samples, not full pages."),
         "parameters": {"type": "object",
                        "properties": {"query": {"type": "string",
                                                 "description": "keyword search query"}},
                        "required": [___]}}},
    {"type": "function",
     "function": {
         "name": "___",
         "description": "Read the FULL text of one page. Use a page_id returned by web_search.",
         "parameters": {"type": "object",
                        "properties": {"page_id": {"type": "string"}},
                        "required": ["page_id"]}}},
]

# Dispatch table: model-visible names -> runtime callables.
TOOL_REGISTRY = {"web_search": web_search, "read_page": ___}

print("Tools the model will know about:", [t["function"]["name"] for t in TOOLS])

<details>
<summary><b>Click here for the solution</b></summary>

```python
# What the model actually sees: JSON schemas, never the Python bodies.
TOOLS = [
    {"type": "function",
     "function": {
         "name": "web_search",
         "description": ("Search an offline copy of the web. Returns the top-3 matching "
                         "pages as [page_id], title, and a 300-character snippet. "
                         "Snippets are samples, not full pages."),
         "parameters": {"type": "object",
                        "properties": {"query": {"type": "string",
                                                 "description": "keyword search query"}},
                        "required": ["query"]}}},
    {"type": "function",
     "function": {
         "name": "read_page",
         "description": "Read the FULL text of one page. Use a page_id returned by web_search.",
         "parameters": {"type": "object",
                        "properties": {"page_id": {"type": "string"}},
                        "required": ["page_id"]}}},
]

# Dispatch table: model-visible names -> runtime callables.
TOOL_REGISTRY = {"web_search": web_search, "read_page": read_page}

print("Tools the model will know about:", [t["function"]["name"] for t in TOOLS])
```

</details>

> **Q:** The lecture says the model "never sees the Python function". What does it see instead, and why does that make tool descriptions prompt engineering?
<details><summary>Click for answer</summary>

The model receives only the JSON tool schema: name, natural-language description, and parameter types. It selects and parameterises tools purely from that text, with no access to the implementation. A vague or misleading description therefore directly causes wrong tool choices and malformed arguments — the description functions as an instruction to the model, exactly like a prompt, and must be written with the same care (including being honest about limitations, e.g. that snippets are samples, not full pages).

</details>

## Part C — The minimal agent loop

Now the poster code of the course, adapted from the lecture's OpenAI-style snippet to
`ollama.chat`. The mapping is almost one-to-one; three Ollama specifics:

- the response message is at `resp["message"]` and has attributes `.content` and `.tool_calls`,
- `call.function.arguments` is **already a dict** — no `json.loads` needed,
- tool results go back as `{"role": "tool", "tool_name": ..., "content": ...}`.

Note where each of the lecture's five design decisions appears: tool schema (Part B), step
cap, error policy, truncation, termination convention. Fill the gaps.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful research agent working against a small offline copy of the web. "
    "Use web_search to find pages and read_page to read a page in full. "
    "When you have gathered enough evidence, give your final answer as plain text "
    "with page_id citations in square brackets."
)

def run_agent(task, max_steps=8, obs_limit=1500,
              system_prompt=SYSTEM_PROMPT, verbose=True):
    """The minimal agent loop: reason, act, observe, decide."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": ___},
    ]
    for step in range(___):   # stop 1: the step cap
        resp = ollama.chat(model=MODEL,
                           messages=___,   # REASON: the FULL history, every time
                           tools=TOOLS)
        msg = resp["message"]
        messages.append(msg)   # state grows here

        if not msg.tool_calls:   # stop 2: final answer (DECIDE is implicit)
            if verbose:
                print(f"[step {step}] FINAL ANSWER")
            return msg.content, messages

        for call in msg.tool_calls:   # ACT: the runtime executes, not the model
            name = call.function.name
            args = call.function.arguments or {}
            if verbose:
                print(f"[step {step}] {name}({args})")
            fn = TOOL_REGISTRY.get(name)
            try:
                result = ___(**args) if fn else f"TOOL ERROR: unknown tool '{name}'"
            except Exception as e:
                result = f"TOOL ERROR: {e}"   # errors become observations
            messages.append({   # OBSERVE
                "role": "tool",
                "tool_name": name,
                "content": str(result)[:___],   # the overflow guard
            })
    else:
        # for-else: runs only when the loop exhausts without returning early
        if verbose:
            print("Step budget exhausted, no answer.")
        return None, messages

<details>
<summary><b>Click here for the solution</b></summary>

```python
SYSTEM_PROMPT = (
    "You are a careful research agent working against a small offline copy of the web. "
    "Use web_search to find pages and read_page to read a page in full. "
    "When you have gathered enough evidence, give your final answer as plain text "
    "with page_id citations in square brackets."
)

def run_agent(task, max_steps=8, obs_limit=1500,
              system_prompt=SYSTEM_PROMPT, verbose=True):
    """The minimal agent loop: reason, act, observe, decide."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    for step in range(max_steps):   # stop 1: the step cap
        resp = ollama.chat(model=MODEL,
                           messages=messages,   # REASON: the FULL history, every time
                           tools=TOOLS)
        msg = resp["message"]
        messages.append(msg)   # state grows here

        if not msg.tool_calls:   # stop 2: final answer (DECIDE is implicit)
            if verbose:
                print(f"[step {step}] FINAL ANSWER")
            return msg.content, messages

        for call in msg.tool_calls:   # ACT: the runtime executes, not the model
            name = call.function.name
            args = call.function.arguments or {}
            if verbose:
                print(f"[step {step}] {name}({args})")
            fn = TOOL_REGISTRY.get(name)
            try:
                result = fn(**args) if fn else f"TOOL ERROR: unknown tool '{name}'"
            except Exception as e:
                result = f"TOOL ERROR: {e}"   # errors become observations
            messages.append({   # OBSERVE
                "role": "tool",
                "tool_name": name,
                "content": str(result)[:obs_limit],   # the overflow guard
            })
    else:
        # for-else: runs only when the loop exhausts without returning early
        if verbose:
            print("Step budget exhausted, no answer.")
        return None, messages
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

Line by line:

- **`messages = [...]`** — the agent's entire working state. The system prompt holds standing instructions, the user message the task. Everything that happens is *appended* here; nothing is ever removed.
- **`for step in range(max_steps)`** — the very first line of the loop is already a safety mechanism. Nothing else guarantees termination.
- **`ollama.chat(..., messages=messages, ...)`** — the *full* history is resent every iteration. This one line is the reason for the statelessness discussion and for the quadratic cost curve of Part D.
- **`messages.append(msg)`** — the model's own outputs (reasoning, tool-call requests) become part of the context it sees next time. The agent reads its own past thoughts the same way it reads tool results.
- **`if not msg.tool_calls`** — the implicit *decide* phase: no explicit branch in the model, the output type is the control signal. Plain text = cooperative exit.
- **dispatch via `TOOL_REGISTRY`** — a name-based routing table from model-visible names to runtime callables; unknown names are handled as errors fed back to the model.
- **`try/except → "TOOL ERROR: ..."`** — the error *policy*: exceptions become observations the model can read and react to (retry, rephrase, route around). A policy choice, not a technical necessity.
- **`str(result)[:obs_limit]`** — the crudest possible context management: a positional cut that protects the window and silently destroys information.
- **`for`–`else`** — Python runs the `else` branch when the loop exhausts without an early exit: the honest-failure path of the step-cap stop condition.

</details>

In [ ]:
# The research agent's first real task this semester:
task = ("What obligations does the EU AI Act place on providers of "
        "general-purpose AI models? Write a short, sourced summary.")

answer, transcript = run_agent(___)

print("\n=== ANSWER ===")
print(answer)
print(f"\nmessages in final context: {len(___)}")

<details>
<summary><b>Click here for the solution</b></summary>

```python
# The research agent's first real task this semester:
task = ("What obligations does the EU AI Act place on providers of "
        "general-purpose AI models? Write a short, sourced summary.")

answer, transcript = run_agent(task)

print("\n=== ANSWER ===")
print(answer)
print(f"\nmessages in final context: {len(transcript)}")
```

</details>

> **Q:** What does it mean that the LLM is stateless, and where does the agent's state live instead?
<details><summary>Click for answer</summary>

Between API calls the model retains nothing: each request is processed from scratch, as a pure function from context tokens to output tokens. The agent's entire working state — task, history of actions, observations, intermediate conclusions — lives in the message list maintained by the loop in the runtime. The "agent" is therefore the loop plus its data, not the model.

</details>

> **Q:** Why does the model only learn the result of a tool call on the *next* loop iteration?
<details><summary>Click for answer</summary>

An API call is a single, atomic completion: the model produces the tool call and the response ends. The runtime then executes the tool and appends the result to the message list. Only when the runtime issues the next API call — resending the grown list — does the model process the observation. There is no channel for pushing data into a completed call.

</details>

> **Q:** Why is "plain text means done" a fragile termination convention, and what is the standard remedy?
<details><summary>Click for answer</summary>

Models frequently emit conversational filler ("Let me now search for...") without a tool call; the loop misreads this as a final answer and stops mid-task. The standard remedy is an explicit finish signal: a dedicated `finish` / `submit_answer` tool the model must call deliberately, optionally with the answer as an argument, so termination requires a structured, intentional act.

</details>

> **📝 Report task R1:** For each of the four stop-condition families — final answer, step cap, budget cap, human interrupt — state *whose judgement* ends the run. Then explain in 3–5 sentences why a production agent keeps **all four** active simultaneously ("defence in depth"), naming the characteristic way each one fails.
>
> *No solution is provided — include your answer/code and a short justification in your lab report.*

## Part D — Instrumentation: watching the loop work

*"Build the thirty-line agent, break it three ways, and instrument it."* — instrumentation
first, so we can *see* the breakage in Part E. We log one row per iteration:

- `context_chars` — a size proxy: total characters of message content currently in the list,
- `prompt_tokens` / `output_tokens` — Ollama reports `prompt_eval_count` and `eval_count`
  per response,
- `latency_s` and the chosen `action`.

One subtlety worth knowing **before** you look at the numbers: Ollama keeps a KV-cache of
the unchanged prefix between calls, so `prompt_eval_count` may count only *newly evaluated*
tokens rather than the full context. If you see small, flat values, you are watching
**prompt caching** — the lecture's cost lever #1 — happen live. The `context_chars` column
shows the true (monotonically growing) context size either way.

In [ ]:
def context_chars(messages):
    """Rough size proxy: total characters of message content in the context."""
    total = 0
    for m in messages:
        content = m["content"] if isinstance(m, dict) else (m.content or "")
        total += len(content or "")
    return total

def run_agent_logged(task, max_steps=8, obs_limit=1500,
                     system_prompt=SYSTEM_PROMPT):
    """The same loop as run_agent, but every iteration writes one log row."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    log, answer = [], None
    for step in range(max_steps):
        t0 = time.time()
        resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
        msg = resp["message"]
        messages.append(msg)
        calls = msg.tool_calls or []
        log.append({
            "step":          step,
            "context_chars": ___(messages),   # our size proxy
            "prompt_tokens": resp.get("___", 0),   # input tokens Ollama evaluated
            "output_tokens": resp.get("eval_count", 0),
            "latency_s":     round(time.time() - t0, 2),
            "action":        calls[0].function.name if calls else "___",
        })
        if not calls:
            answer = msg.content
            break
        for call in calls:
            args = call.function.arguments or {}
            fn = TOOL_REGISTRY.get(call.function.name)
            if fn is None:
                result = f"TOOL ERROR: unknown tool '{call.function.name}'"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"TOOL ERROR: {e}"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    return answer, messages, pd.DataFrame(log)

<details>
<summary><b>Click here for the solution</b></summary>

```python
def context_chars(messages):
    """Rough size proxy: total characters of message content in the context."""
    total = 0
    for m in messages:
        content = m["content"] if isinstance(m, dict) else (m.content or "")
        total += len(content or "")
    return total

def run_agent_logged(task, max_steps=8, obs_limit=1500,
                     system_prompt=SYSTEM_PROMPT):
    """The same loop as run_agent, but every iteration writes one log row."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    log, answer = [], None
    for step in range(max_steps):
        t0 = time.time()
        resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
        msg = resp["message"]
        messages.append(msg)
        calls = msg.tool_calls or []
        log.append({
            "step":          step,
            "context_chars": context_chars(messages),   # our size proxy
            "prompt_tokens": resp.get("prompt_eval_count", 0),   # input tokens Ollama evaluated
            "output_tokens": resp.get("eval_count", 0),
            "latency_s":     round(time.time() - t0, 2),
            "action":        calls[0].function.name if calls else "FINAL",
        })
        if not calls:
            answer = msg.content
            break
        for call in calls:
            args = call.function.arguments or {}
            fn = TOOL_REGISTRY.get(call.function.name)
            if fn is None:
                result = f"TOOL ERROR: unknown tool '{call.function.name}'"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"TOOL ERROR: {e}"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    return answer, messages, pd.DataFrame(log)
```

</details>

In [ ]:
answer, transcript, log = run_agent_logged(task)
print(log.to_string(index=False))

cum_input = log["prompt_tokens"].___()   # total input tokens paid so far

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(log["step"], log["context_chars"], "o-")
axes[0].set(title="context size per step", xlabel="step", ylabel="chars")
axes[1].plot(log["step"], ___, "o-", color="crimson")
axes[1].set(title="cumulative input tokens paid", xlabel="step", ylabel="tokens")
plt.tight_layout()
plt.show()

print(f"final context: {log['prompt_tokens'].iloc[-1]:.0f} tokens per last call; "
      f"cumulative input paid: {cum_input.iloc[-1]:.0f} tokens")

<details>
<summary><b>Click here for the solution</b></summary>

```python
answer, transcript, log = run_agent_logged(task)
print(log.to_string(index=False))

cum_input = log["prompt_tokens"].cumsum()   # total input tokens paid so far

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(log["step"], log["context_chars"], "o-")
axes[0].set(title="context size per step", xlabel="step", ylabel="chars")
axes[1].plot(log["step"], cum_input, "o-", color="crimson")
axes[1].set(title="cumulative input tokens paid", xlabel="step", ylabel="tokens")
plt.tight_layout()
plt.show()

print(f"final context: {log['prompt_tokens'].iloc[-1]:.0f} tokens per last call; "
      f"cumulative input paid: {cum_input.iloc[-1]:.0f} tokens")
```

</details>

**Reading the plots.** The context grows roughly *linearly* per step (left), so the
*cumulative* input paid grows roughly *quadratically* (right) — unless Ollama's prefix cache
flattens `prompt_tokens`, in which case you are seeing exactly why providers built prompt
caching for loop-shaped workloads. Run the cell a second time in the same session and
compare.

> **Q:** Why do output tokens grow only linearly while input tokens grow quadratically?
<details><summary>Click for answer</summary>

Each thought or tool call is generated exactly once, so output is the sum of per-step generations — linear in steps. But every generated or observed token is re-read as input on all subsequent iterations, so early tokens are paid for repeatedly; summing the re-reads yields the quadratic term $k \, n(n+1)/2$.

</details>

> **📝 Report task R2:** Using your log from Part D: (a) estimate the average number of tokens $k$ your agent adds to the context per step; (b) use the lecture's formula $k \, n(n+1)/2$ to predict the cumulative input tokens of a hypothetical 20-step run; (c) compare the formula's prediction for *your* run with the measured cumulative sum and explain any discrepancy (hint: prefix caching); (d) name two cost levers from the lecture and quantify their approximate effect on your numbers.
>
> *No solution is provided — include your answer/code and a short justification in your lab report.*

## Part E — Breaking the loop

The lecture's lab announcement, verbatim: *"You implement exactly this loop, then break it:
remove the step cap, feed it oversized pages, and watch each failure mode appear."*

We now do exactly that. Reminder from the lecture: these failures raise **no exceptions** —
every individual step succeeds; the pathology exists only at the level of the *trajectory*.
LLM runs are stochastic: if an experiment does not misbehave on the first try, run it again.

### Experiment 1 — the runaway loop

We give the agent a task whose answer does **not exist** in our mini-web and effectively
remove the step cap (a generous `max_steps`). Watch it search, fail, and search again.

In [ ]:
# The mini-web contains NOTHING about this task - and we raise the step cap.
doomed_task = ("What final mark does one receive for the Agentic AI practical "
               "sessions in Kiel? Answer only from web sources; do not guess.")

answer, transcript = run_agent(doomed_task, max_steps=___, obs_limit=1500)   # generous cap, e.g. 12

calls = [(c.function.name, str(sorted((c.function.arguments or {}).items())))
         for m in transcript if getattr(m, "tool_calls", None)
         for c in m.tool_calls]
dupes = len(calls) - len(set(___))
print(f"\n{len(calls)} tool calls, {dupes} exact duplicates")
print("answer:", answer)

<details>
<summary><b>Click here for the solution</b></summary>

```python
# The mini-web contains NOTHING about this task - and we raise the step cap.
doomed_task = ("What final mark does one receive for the Agentic AI practical "
               "sessions in Kiel? Answer only from web sources; do not guess.")

answer, transcript = run_agent(doomed_task, max_steps=12, obs_limit=1500)   # generous cap, e.g. 12

calls = [(c.function.name, str(sorted((c.function.arguments or {}).items())))
         for m in transcript if getattr(m, "tool_calls", None)
         for c in m.tool_calls]
dupes = len(calls) - len(set(calls))
print(f"\n{len(calls)} tool calls, {dupes} exact duplicates")
print("answer:", answer)
```

</details>

### Experiment 2 — the step cap as backstop

Same doomed task, tight cap. *"When every other mechanism fails, the for-loop ends because
integers do not hallucinate."* Note the honest-failure exit (`None`) instead of a made-up
answer.

In [ ]:
answer, transcript = run_agent(doomed_task, max_steps=3)
print("answer:", ___)   # None -> the honest-failure (for-else) exit fired

<details>
<summary><b>Click here for the solution</b></summary>

```python
answer, transcript = run_agent(doomed_task, max_steps=3)
print("answer:", answer)   # None -> the honest-failure (for-else) exit fired
```

</details>

> **Q:** Explain why an infinite loop in an agent raises no errors and passes every component test.
<details><summary>Click for answer</summary>

Each individual operation succeeds: the API call returns, the tool executes, the arguments parse. The defect is a property of the *sequence* — the same action repeated with no progress — which no single-call test inspects. Detection therefore requires trajectory-level monitoring, e.g. comparing recent tool calls for duplicates, not component-level assertions.

</details>

### Experiment 3 — context overflow (oversized pages)

`giant_page` is ~60,000 characters of bureaucratic boilerplate with **one** useful fact buried
at about 65% depth. First we switch the overflow guard **off** and feed the whole page into
the context.

In [ ]:
overflow_task = ("Use read_page on the page with page_id 'giant_page' and report "
                 "the annual budget of the Bureau of Very Long Documents.")

answer, transcript = run_agent(overflow_task, max_steps=4, obs_limit=___)   # guard OFF: try 200_000
print(f"\ncontext size: {context_chars(transcript):,} chars "
      f"(≈ {context_chars(transcript) // 4:,} tokens)")
print("answer:", str(answer)[:400])

<details>
<summary><b>Click here for the solution</b></summary>

```python
overflow_task = ("Use read_page on the page with page_id 'giant_page' and report "
                 "the annual budget of the Bureau of Very Long Documents.")

answer, transcript = run_agent(overflow_task, max_steps=4, obs_limit=200_000)   # guard OFF: try 200_000
print(f"\ncontext size: {context_chars(transcript):,} chars "
      f"(≈ {context_chars(transcript) // 4:,} tokens)")
print("answer:", str(answer)[:400])
```

</details>

In [ ]:
# Now with the overflow guard back on - what does truncation destroy?
answer, transcript = run_agent(overflow_task, max_steps=4, obs_limit=___)   # guard back ON: 1500
print(f"context size: {context_chars(transcript):,} chars")
print("answer:", str(answer)[:400])

<details>
<summary><b>Click here for the solution</b></summary>

```python
# Now with the overflow guard back on - what does truncation destroy?
answer, transcript = run_agent(overflow_task, max_steps=4, obs_limit=1500)   # guard back ON: 1500
print(f"context size: {context_chars(transcript):,} chars")
print("answer:", str(answer)[:400])
```

</details>

**What you should observe.** With the guard off, the ~60k-character page (≈ 15k tokens)
usually exceeds the model's context window as configured in Ollama (`num_ctx`, typically
4k–8k by default). Ollama then **silently truncates** the prompt — the lecture's *silent*
overflow variant: no error, but the oldest tokens (system prompt! task!) can drop, latency
jumps, and the answer degrades or confabulates. With the guard back on (`obs_limit=1500`),
the context stays healthy — but the positional cut lands *before* the buried fact at 65%
depth, so the agent cannot find the budget figure at all: `result[:obs_limit]` *"protects the
context window — and silently destroys information."* Neither setting is "correct"; that
tension is what Unit 6 (memory & context management) is about.

> **Q:** Describe the two distinct ways context overflow can manifest, and why the silent variant is more dangerous.
<details><summary>Click for answer</summary>

Hard variant: the request exceeds the model's context limit and the API returns an error — loud, immediate, debuggable. Silent variant: some layer (here: Ollama itself) evicts or truncates the oldest tokens to fit, which typically includes the system prompt and the original task. The agent keeps acting with its goal and constraints amputated — it does not stop, it acts *wrongly*, which is far harder to detect and potentially harmful.

</details>

### Experiment 4 — derailment bait

Our mini-web contains a page about **US executive orders** that EU-AI-Act searches also
surface (it mentions the EU AI Act repeatedly — just like the lecture's case study, where one
retrieved source about US policy derailed the research agent from step 4 onward). Run the
research task several times and *read the trajectories*: does the agent ever start chasing US
policy instead of GPAI obligations?

In [ ]:
for run in range(___):   # a few runs, e.g. 3 - trajectories differ between runs
    answer, transcript = run_agent(task, max_steps=8, verbose=False)
    actions = []
    for m in transcript:
        for c in (getattr(m, "tool_calls", None) or []):
            arg = (c.function.arguments or {})
            actions.append(f"{c.function.name}({arg.get('query') or arg.get('page_id')})")
    print(f"run {run}: " + " -> ".join(actions))

<details>
<summary><b>Click here for the solution</b></summary>

```python
for run in range(3):   # a few runs, e.g. 3 - trajectories differ between runs
    answer, transcript = run_agent(task, max_steps=8, verbose=False)
    actions = []
    for m in transcript:
        for c in (getattr(m, "tool_calls", None) or []):
            arg = (c.function.arguments or {})
            actions.append(f"{c.function.name}({arg.get('query') or arg.get('page_id')})")
    print(f"run {run}: " + " -> ".join(actions))
```

</details>

## Part F — First-line defences

The lecture's starter kit (full guardrail engineering follows in Unit 7):

| Failure mode | Early symptom | First-line defence |
|---|---|---|
| Infinite loop | identical tool call repeated across steps | step cap; **duplicate-call detector** |
| Context overflow | latency and cost rise every step | truncate observations; summarise old turns |
| Derailment | actions stop serving the stated goal | **re-inject goal**; plan checkpoints; human review |

We also add the missing third stop condition: a **budget cap** — *"the cost model and the
stop condition are the same mechanism seen twice."* Fill the gaps in the budget-cap loop;
the two defences afterwards are report tasks.

In [ ]:
def run_agent_v2(task, max_steps=12, obs_limit=1500, token_budget=20_000,
                 system_prompt=SYSTEM_PROMPT, verbose=True):
    """The minimal loop + stop condition 3: a cumulative token budget."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    spent = 0
    for step in range(max_steps):
        resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
        spent += resp.get("prompt_eval_count", 0) + resp.get("___", 0)   # + output tokens
        msg = resp["message"]
        messages.append(msg)
        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] FINAL ANSWER ({spent} tokens spent)")
            return msg.content, messages
        if spent > ___:   # stop 3: budget cap
            if verbose:
                print(f"[step {step}] budget exhausted ({spent} tokens) - stopping")
            return None, messages
        for call in msg.tool_calls:
            args = call.function.arguments or {}
            fn = TOOL_REGISTRY.get(call.function.name)
            if fn is None:
                result = f"TOOL ERROR: unknown tool '{call.function.name}'"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"TOOL ERROR: {e}"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    if verbose:
        print("Step budget exhausted, no answer.")
    return None, messages

answer, transcript = run_agent_v2(task, token_budget=___)   # try a tight budget, e.g. 6_000
print("answer:", str(answer)[:300])

<details>
<summary><b>Click here for the solution</b></summary>

```python
def run_agent_v2(task, max_steps=12, obs_limit=1500, token_budget=20_000,
                 system_prompt=SYSTEM_PROMPT, verbose=True):
    """The minimal loop + stop condition 3: a cumulative token budget."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    spent = 0
    for step in range(max_steps):
        resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
        spent += resp.get("prompt_eval_count", 0) + resp.get("eval_count", 0)   # + output tokens
        msg = resp["message"]
        messages.append(msg)
        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] FINAL ANSWER ({spent} tokens spent)")
            return msg.content, messages
        if spent > token_budget:   # stop 3: budget cap
            if verbose:
                print(f"[step {step}] budget exhausted ({spent} tokens) - stopping")
            return None, messages
        for call in msg.tool_calls:
            args = call.function.arguments or {}
            fn = TOOL_REGISTRY.get(call.function.name)
            if fn is None:
                result = f"TOOL ERROR: unknown tool '{call.function.name}'"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"TOOL ERROR: {e}"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    if verbose:
        print("Step budget exhausted, no answer.")
    return None, messages

answer, transcript = run_agent_v2(task, token_budget=6_000)   # try a tight budget, e.g. 6_000
print("answer:", str(answer)[:300])
```

</details>

> **📝 Report task R3:** Complete the cell below — a **duplicate-call detector** against infinite loops. Normalise each tool call to a hashable key (tool name + sorted, lower-cased arguments), remember it, and on a repeat do **not** execute the tool: instead feed back the intervention message. Test it on `doomed_task` and include in your report: your code, one run where the detector intervenes, and 2–3 sentences on why detection must happen at *trajectory* level.
>
> *No solution is provided — include your answer/code and a short justification in your lab report.*

In [ ]:
def run_agent_v3(task, max_steps=12, obs_limit=1500,
                 system_prompt=SYSTEM_PROMPT, verbose=True):
    """v2's idea + a first-line defence against infinite loops:
    a duplicate-call detector (lecture: 'detect exactly that')."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    seen = ___   # remembers normalised calls
    for step in range(max_steps):
        resp = ollama.chat(model=MODEL, messages=messages, tools=TOOLS)
        msg = resp["message"]
        messages.append(msg)
        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] FINAL ANSWER")
            return msg.content, messages
        for call in msg.tool_calls:
            args = call.function.arguments or {}
            key = ___   # normalise
            if key in seen:
                result = ("You already executed this exact call; the result will not "
                          "change. Choose a different action or give your final answer.")
                if verbose:
                    print(f"[step {step}] DUPLICATE blocked: {call.function.name}({args})")
            else:
                seen.___   # remember this call
                fn = TOOL_REGISTRY.get(call.function.name)
                if fn is None:
                    result = f"TOOL ERROR: unknown tool '{call.function.name}'"
                else:
                    try:
                        result = fn(**args)
                    except Exception as e:
                        result = f"TOOL ERROR: {e}"
                if verbose:
                    print(f"[step {step}] {call.function.name}({args})")
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    if verbose:
        print("Step budget exhausted, no answer.")
    return None, messages

# Test on the doomed task - the detector should intervene:
answer, transcript = run_agent_v3(doomed_task, max_steps=8)

> **📝 Report task R4:** Complete the cell below — **goal re-injection** against derailment (the lecture's fix for the EU-AI-Act case study: *"restate the goal every iteration"*). The reminder must be appended to the *request* only, not to the stored history. Run the comparison at the bottom and include in your report: your code, the printed trajectories, and 2–3 sentences on what changed (expect a *distributional* improvement, not perfection) and what the re-injection costs per step.
>
> *No solution is provided — include your answer/code and a short justification in your lab report.*

In [ ]:
def run_agent_v4(task, max_steps=8, obs_limit=1500, reinject=True,
                 system_prompt=SYSTEM_PROMPT, verbose=True):
    """The minimal loop + a first-line defence against derailment: goal re-injection.
    The goal is restated as the LAST message of every request, so it is never
    thousands of tokens away from the model's attention."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": task},
    ]
    reminder = {"role": "user",
                "content": f"REMINDER - your one and only goal: {task}"}
    for step in range(max_steps):
        request = ___   # history, plus the reminder if reinject is on
        resp = ollama.chat(model=MODEL, messages=request, tools=TOOLS)
        msg = resp["message"]
        messages.append(___)   # NB: grow messages, not request
        if not msg.tool_calls:
            if verbose:
                print(f"[step {step}] FINAL ANSWER")
            return msg.content, messages
        for call in msg.tool_calls:
            args = call.function.arguments or {}
            fn = TOOL_REGISTRY.get(call.function.name)
            if fn is None:
                result = f"TOOL ERROR: unknown tool '{call.function.name}'"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"TOOL ERROR: {e}"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    if verbose:
        print("Step budget exhausted, no answer.")
    return None, messages

# Compare trajectories with and without re-injection (3 runs each):
for reinject in (False, True):
    print(f"\n--- reinject={reinject} ---")
    for run in range(3):
        answer, transcript = run_agent_v4(task, reinject=reinject, verbose=False)
        actions = []
        for m in transcript:
            for c in (getattr(m, "tool_calls", None) or []):
                arg = (c.function.arguments or {})
                actions.append(f"{c.function.name}({arg.get('query') or arg.get('page_id')})")
        print(f"run {run}: {len(actions)} tool calls -> " + " | ".join(actions))

## Part G — Tuning & exploration

No gaps in this part — just knobs. The `explore` function below is the full minimal loop
once more, with the interesting parameters exposed. Things worth trying (one knob at a time):

- **`max_steps`** 2 vs. 8 vs. 20 — when does the cap bite, when is it slack?
- **`obs_limit`** 200 vs. 1500 vs. 5000 — at 200, can the agent still answer the GPAI task,
  or does truncation destroy the obligations before the model sees them?
- **`temperature`** 0.0 vs. 0.7 vs. 1.3 — how much do trajectories vary between runs?
  Does a hotter model derail toward the US-policy page more often?
- **`model`** — pull `qwen2.5:3b` and compare: does a smaller model loop or derail more?
- **tasks** — try `doomed_task`, `overflow_task`, or invent your own.

In [ ]:
def explore(task=task, model=MODEL, max_steps=8, obs_limit=1500, temperature=0.7):
    """Playground: the minimal loop with its knobs exposed. No gaps here - just play."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": task}]
    for step in range(max_steps):
        resp = ollama.chat(model=model, messages=messages, tools=TOOLS,
                           options={"temperature": temperature})
        msg = resp["message"]
        messages.append(msg)
        if not msg.tool_calls:
            print(f"[step {step}] FINAL ({context_chars(messages):,} chars in context)")
            print(msg.content)
            return
        for call in msg.tool_calls:
            args = call.function.arguments or {}
            print(f"[step {step}] {call.function.name}({args})")
            fn = TOOL_REGISTRY.get(call.function.name)
            if fn is None:
                result = f"TOOL ERROR: unknown tool '{call.function.name}'"
            else:
                try:
                    result = fn(**args)
                except Exception as e:
                    result = f"TOOL ERROR: {e}"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)[:obs_limit]})
    print(f"Step budget exhausted ({context_chars(messages):,} chars in context).")

explore(max_steps=6, obs_limit=800, temperature=0.2)

In [ ]:
# Optional: interactive knobs (safe to skip if ipywidgets is not installed).
try:
    import ipywidgets as widgets
    from ipywidgets import interact_manual

    interact_manual(
        explore,
        task=widgets.Text(value=task, description="task"),
        model=widgets.Text(value=MODEL, description="model"),
        max_steps=widgets.IntSlider(min=1, max=20, value=8, description="max_steps"),
        obs_limit=widgets.IntSlider(min=100, max=8000, step=100, value=1500,
                                    description="obs_limit"),
        temperature=widgets.FloatSlider(min=0.0, max=1.5, step=0.1, value=0.7,
                                        description="temperature"),
    )
    print("Adjust the knobs, then press 'Run Interact'.")
except ImportError:
    print("ipywidgets is not installed - call explore(...) by hand instead, e.g.:")
    print("  explore(max_steps=12, obs_limit=300, temperature=1.2)")

> **Q:** Name the four agentic design patterns in Ng's taxonomy and give a one-sentence mechanism for each.
<details><summary>Click for answer</summary>

**Planning:** decompose the goal into explicit subgoals before acting and execute against the plan, replanning on new evidence. **Tool Use:** extend the model with external actions via structured calls executed by the runtime. **Reflection:** have the agent critique its own output and revise it in a draft–review–redraft cycle. **Multi-Agent:** split the task across specialised agents with separate contexts that collaborate via messages. None of them changes the loop itself — they are architectures *of* loops.

</details>

> **📝 Report task R5:** (a) Our minimal agent already implements one of Ng's four design patterns — which one, and what exactly in the code qualifies it? (b) For the *research agent*, which single additional pattern would you add first to improve the final report, and why? Justify in about five sentences, referring to at least one experiment from this lab.
>
> *No solution is provided — include your answer/code and a short justification in your lab report.*

## Wrap-up

**Takeaways.**

- The loop is the agent: reason → act → observe → decide, around a model that executes nothing itself. You built it in ~30 lines around `ollama.chat`.
- State is the message list — the model is stateless; the agent's memory, identity and progress are data in *your* hands.
- Stops are designed, not assumed: final answer, step cap, budget cap (you added it), human interrupt. The step cap is the only one that needs no cooperation.
- The failures you produced — runaway repetition, silent context overflow, derailment bait — are emergent properties of iteration; the unit of failure is the trajectory, not the call.
- Cumulative input grows as $k \, n(n+1)/2$ — you measured the quadratic tax and (probably) caught prompt caching flattening it.
- Four patterns ahead: Planning, Tool Use (done!), Reflection, Multi-Agent.

**Next week (Session 03 — Tools):** function calling mechanics token by token, what makes a
tool description good, and the Model Context Protocol — the interface you hand-declared in
Part B, done properly and standardised.

---

### 📋 For your lab report

| # | Task | Where |
|---|---|---|
| R1 | Four stop-condition families: whose judgement, and why defence in depth needs all four | Part C |
| R2 | Quadratic-tax estimate from your own log: $k$, 20-step prediction, formula vs. measurement, two cost levers | Part D |
| R3 | Duplicate-call detector: completed code + an intervening run + why detection is trajectory-level | Part F |
| R4 | Goal re-injection: completed code + trajectory comparison + cost of the reminder | Part F |
| R5 | Which pattern the minimal agent already implements; which one you would add first and why | Part G |

Include short justifications, not just code and numbers. Good luck — and remember:
*integers do not hallucinate.*